In [1]:
import os
import json
import requests
from dotenv import load_dotenv
from google.oauth2 import service_account
from google.cloud import storage
from google.cloud import bigquery
from io import BytesIO


load_dotenv()

True

In [ ]:
url = "TOKEN_URL"

headers = {
    "Accept": "application/json",
    "Content-Type": "application/json"
}

data = {
    "client_id": "CLIENT_ID",
    "client_secret": "CLIENT_SECRET",
    "scope": "read_orders",
    "grant_type": "client_credentials"
}

response = requests.post(url, headers=headers, json=data)

token = response.json()["data"]["access_token"]

orders_url = "ORDERS_URL"

orders_headers = {
    "Accept": "application/json",
    "Authorization": f"Bearer {token}"
}


# looping over all pages of data for timeframe
pageIndex = 0
all_results = []


while True:
    params = {
        "endDate": "2026-01-31",
        "pageIndex": pageIndex,
        "startDate": "2026-01-01"
    }
    
    orders_response = requests.get(orders_url, headers=orders_headers, params=params)

    if orders_response.status_code != 200:
        print("Request failed:", orders_response.text)
        break

    orders_data = orders_response.json()
    items = orders_data.get("data", [])

    if not items:
        break

    all_results.extend(items)
    pageIndex += 1


# converting to ndjson
ndjson_lines = [json.dumps(record) for record in all_results]
ndjson_content = "\n".join(ndjson_lines)


project_id = os.getenv("GCP_PROJECT_ID")
service_account_key = os.getenv("GCP_SERVICE_ACCOUNT_KEY")

credentials = service_account.Credentials.from_service_account_file(service_account_key)
client = bigquery.Client(project=project_id, credentials=credentials)

dataset_name = "rocket_rez_data"
table_name = "raw_data"

table_id = f"{project_id}.{dataset_name}.{table_name}"

ndjson_bytes = ndjson_content.encode("utf-8")
buffer = BytesIO(ndjson_bytes)

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
    write_disposition="WRITE_TRUNCATE",  # WRITE_TRUNCATE -> deletes all existing rows and replaces them with your new data
    ignore_unknown_values=True
)

load_job = client.load_table_from_file(
    buffer,
    table_id,
    job_config=job_config
)

load_job.result()

LoadJob<project=rocket-rez-api, location=us-central1, id=55699758-e9b4-4198-819b-e763c76f53c9>

In [3]:
print(orders_response.status_code)

200
